# SID-MPC Load Tracking with W&B Logging

This notebook runs an MPC controller that uses:

1. the learned thermal SID model from `saved_files/advanced_sid`,
2. approximate known battery/SOC dynamics,
3. approximate algebraic net-load balance,
4. comfort penalties for indoor temperature.

It logs district-load tracking, KPIs, and temperature plots to W&B.

Before running this notebook:

- run `CityLearn_Advanced_SID_Notebook.ipynb` and save the SID artifacts;
- copy `sid_mpc_controller.py` into `MPC/MPC_Advanced_SID/sid_mpc_controller.py`;
- select the correct kernel, e.g. `Python (citylearn_berkan)`.


In [1]:
import os
os.environ["WANDB_SILENT"] = "true"

from pathlib import Path
import sys, warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import wandb
import plotly.graph_objects as go

WANDB_ENTITY = "CityLearn-TeamB"
WANDB_PROJECT = "CityLearn"  
RUN_NAME = "MPC_Advanced_SID"

## 1. Locate repo, dataset, and local modules


In [2]:
HERE = Path.cwd()
REPO_DIR = None
for p in [HERE] + list(HERE.parents):
    if (p / "citylearn").exists():
        REPO_DIR = p
        break

if REPO_DIR is None:
    raise FileNotFoundError("Could not find repo root containing citylearn/. Open this notebook inside the project repo.")

sys.path.insert(0, str(REPO_DIR))

CLIMATE = "TX"
DATASET_DIR = REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
if not (DATASET_DIR / "schema.json").exists():
    DATASET_DIR = REPO_DIR / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

SID_DIR = REPO_DIR / "System Identification" / "advanced_sid"
MPC_MODULE_DIR = REPO_DIR / "MPC" / "MPC_Advanced_SID"
MPC_MODULE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(MPC_MODULE_DIR))

print("REPO_DIR:", REPO_DIR)
print("DATASET_DIR:", DATASET_DIR, DATASET_DIR.exists())
print("SCHEMA_PATH:", SCHEMA_PATH, SCHEMA_PATH.exists())
print("SID_DIR:", SID_DIR, SID_DIR.exists())
print("MPC_MODULE_DIR:", MPC_MODULE_DIR, MPC_MODULE_DIR.exists())

REPO_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1
DATASET_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood True
SCHEMA_PATH: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood\schema.json True
SID_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\System Identification\advanced_sid True
MPC_MODULE_DIR: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\MPC\MPC_Advanced_SID True


## 2. Import CityLearn and SID-MPC controller


In [3]:
from citylearn.citylearn import CityLearnEnv
from sid_mpc_controller import LearnedThermalSID, SIDMPCConfig, SIDMPCController

print("Imports OK")


Imports OK


## 3. Load district target and create environment


In [4]:
def make_env(central_agent=False):
    return CityLearnEnv(schema=str(SCHEMA_PATH), root_directory=str(DATASET_DIR), central_agent=central_agent)

# TX - June window
sim_start = 3624
sim_end = 4343

# Use the target file if present. Otherwise fall back to env district target if your fork exposes it.
district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values[sim_start:sim_end + 1]
T_total = len(district_target)

print("Simulation window:", sim_start, sim_end, T_total)
print("Target shape:", district_target.shape)
print("Target min/mean/max:", np.min(district_target), np.mean(district_target), np.max(district_target))

env = make_env(central_agent=False)
obs, info = env.reset()
print("Number of buildings:", len(env.buildings))
print("Action names first building:", env.action_names[0] if isinstance(env.action_names[0], list) else env.action_names[:2])


Simulation window: 3624 4343 720
Target shape: (720,)
Target min/mean/max: 16.984078912583332 19.50426806551711 23.366495531833333
Number of buildings: 25
Action names first building: ['electrical_storage', 'cooling_device']


## 4. Load learned SID model and configure MPC

Start with horizon 8 or 12. Increase only after the controller runs successfully.


In [5]:
sid_model = LearnedThermalSID(SID_DIR)
print("SID features:", len(sid_model.features))
print("SID device:", sid_model.device)
print("Residual models:", len(sid_model.models))

mpc_config = SIDMPCConfig(
    horizon=4,          # start small; 12-24 is slower
    comfort_low=22.0,
    comfort_high=26.0,
    w_track=80.0,
    w_comfort=20.0,
    w_smooth=1.0,
    w_soc=1.0,
    terminal_soc_ref=0.50,
    maxiter=40,
    use_slsqp=True,
    verbose=False,
)

agent = SIDMPCController(env, sid_model, district_target=district_target, config=mpc_config)
print("MPC controller ready")


SID features: 30
SID device: cuda
Residual models: 3
MPC controller ready


## 5. Run SID-MPC and log district load to W&B


In [6]:
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=RUN_NAME,
    reinit=True,
    settings=wandb.Settings(console="off")
)

wandb.define_metric("hour")
wandb.define_metric("district_load", step_metric="hour")

# Optional speed limit for debugging. Set to None for full 720h.
MAX_RUN_STEPS = None  # e.g. 48 for quick debug

with tqdm(total=T_total if MAX_RUN_STEPS is None else min(T_total, MAX_RUN_STEPS), desc="SID-MPC", unit="step") as pbar:

    while not env.terminated:

        if MAX_RUN_STEPS is not None and len(env.net_electricity_consumption) >= MAX_RUN_STEPS:
            break

        actions = agent.predict(obs)
        obs, _, terminated, truncated, _ = env.step(actions)

        # Get latest district load for live W&B logging
        current_load = env.net_electricity_consumption[-1]

        t = len(env.net_electricity_consumption) - 1

        wandb.log({
            "hour": t,
            "district_load": float(current_load),
        })

        pbar.update(1)

        if terminated or truncated:
            break

# Extract final arrays from environment after simulation
district_load_sid_mpc = np.array(env.net_electricity_consumption)

sid_mpc_building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:len(district_load_sid_mpc)]
    for b in env.buildings
]).T

# Match target length to actual simulation length
district_target_eval = district_target[:len(district_load_sid_mpc)]

print("district_load_sid_mpc shape:", district_load_sid_mpc.shape)
print("sid_mpc_building_temps shape:", sid_mpc_building_temps.shape)

SID-MPC:   0%|          | 0/720 [00:00<?, ?step/s]

KeyboardInterrupt: 

## 6. Extract temperatures and compute KPIs


In [7]:
from SERVER.KPIs import compute_kpis

kpis_sid_mpc = compute_kpis(district_target_eval, district_load_sid_mpc, sid_mpc_building_temps)

wandb.log({
    "NMBE [%]": float(kpis_sid_mpc["NMBE [%]"]),
    "CV-RMSE [%]": float(kpis_sid_mpc["CV-RMSE [%]"]),
    "Temp Comfort violation [%]": float(kpis_sid_mpc["Temp Comfort violation [%]"]),
})

print(kpis_sid_mpc)

{'NMBE [%]': 18.478, 'CV-RMSE [%]': 65.319, 'Temp Comfort violation [%]': 15.33}


## 7. Log load-tracking plot


## 8. Log temperature comfort plot


In [8]:
T = sid_mpc_building_temps.shape[0]

fig = go.Figure()

# Comfort band 22–26 °C
fig.add_hrect(
    y0=22,
    y1=26,
    fillcolor="green",
    opacity=0.12,
    line_width=0,
    annotation_text="Comfort band [22–26 °C]",
    annotation_position="top right"
)

fig.add_hline(y=22, line_dash="dash", line_color="red", line_width=1)
fig.add_hline(y=26, line_dash="dash", line_color="red", line_width=1)

# Individual building temperatures
for b in range(sid_mpc_building_temps.shape[1]):

    fig.add_trace(go.Scatter(
        x=list(range(T)),
        y=sid_mpc_building_temps[:, b],
        mode="lines",
        line=dict(color="rgba(70,130,180,0.2)", width=1),
        showlegend=False
    ))

# Mean temperature
mean_temp = sid_mpc_building_temps.mean(axis=1)

fig.add_trace(go.Scatter(
    x=list(range(T)),
    y=mean_temp,
    mode="lines",
    line=dict(color="steelblue", width=3),
    name="Mean temperature"
))

fig.update_layout(
    title="SID-MPC - Indoor Temperature",
    xaxis_title="Hour",
    yaxis_title="Temperature [°C]"
)

wandb.log({
    "temperature_plot": wandb.Plotly(fig)
})

## 9. Save arrays locally and finish W&B


In [9]:
wandb.finish()